<a href="https://colab.research.google.com/github/nikolas-joyce/macro-signal-strategies/blob/main/08-investment-clock/notebooks/02_signal_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Investment Clock - Notebook 2: Signal Construction


## 1. Load Aligned Panel


In [ ]:
# Cell 1: Drive mount (top of every notebook)
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/macro_strategies/'
import os
os.makedirs(BASE_PATH + 'data/raw', exist_ok=True)
os.makedirs(BASE_PATH + 'data/processed', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from src.signal_builder import normalise_signal
panel = pd.read_parquet(BASE_PATH + 'data/processed/aligned_panel.parquet')


## 2. Raw Signal Computation

Implement the strategy's core formula (see `index.md`).


In [ ]:
# Investment Clock: normalise each component signal then equal-weight composite
from src.composite import build_composite

cpi_6m    = panel['CPIAUCSL'].pct_change(6).apply(lambda x: (1 + x)**2 - 1) * 100
sig_infl  = normalise_signal((cpi_6m - 2.0) / 2.0)
sig_growth = normalise_signal(panel['PAYEMS'].pct_change(12) * 100)
sig_slope  = normalise_signal(panel['GS10'] - panel['GS2'])
sig_credit = normalise_signal(
    panel['TOTLL'].pct_change(4) * 100 - panel['GDPC1'].pct_change(4) * 100
)
sig_m2 = normalise_signal(
    panel['M2SL'].pct_change(12) * 100 - panel['CPIAUCSL'].pct_change(12) * 100
)
components = {
    'inflation': sig_infl, 'growth': sig_growth,
    'yield_curve': sig_slope, 'credit': sig_credit, 'real_m2': sig_m2,
}
raw = pd.DataFrame(components).mean(axis=1)  # equal-weight pre-composite

## 3. Signal Transformations

Rolling z-score, +/- 3 sigma clip, lag-1.


In [ ]:
signal = normalise_signal(raw, window=60)


## 4. Alternative Signal Variants

Headline vs. core, 3m vs. 6m, level vs. change.


In [ ]:
# Investment Clock regime classification
from src.composite import investment_clock_regime, REGIME_ASSET_MAP

growth_sig    = normalise_signal(panel['PAYEMS'].pct_change(12) * 100)
inflation_sig = normalise_signal((panel['CPIAUCSL'].pct_change(6).apply(lambda x: (1+x)**2-1)*100 - 2.0)/2.0)
g = growth_sig.dropna()
inf = inflation_sig.reindex(g.index).dropna()
common_idx = g.index.intersection(inf.index)
regime_series = investment_clock_regime(g.loc[common_idx], inf.loc[common_idx])
print(regime_series.value_counts())
regime_series.value_counts().plot(kind='bar', title='Investment Clock Regime Distribution', figsize=(8, 4))

## 5. Signal Validation

Time-series plot, distribution histogram, autocorrelation.


In [ ]:
signal.plot(figsize=(12, 5), title='Signal (z-score)')


## 6. Predictive Power Pre-Backtest

Rolling Information Coefficient, quantile forward returns.


In [ ]:
import matplotlib.pyplot as plt

mkt_returns = pd.read_parquet(BASE_PATH + 'data/processed/market_returns.parquet')
fwd_returns = mkt_returns.iloc[:, 0].shift(-1)

aligned = pd.concat([signal, fwd_returns], axis=1).dropna()
aligned.columns = ['signal', 'fwd_ret']

ic = aligned['signal'].rolling(12).corr(aligned['fwd_ret'])
fig, ax = plt.subplots(figsize=(12, 4))
ic.plot(ax=ax, title='Rolling 12m Information Coefficient (vs primary asset fwd return)')
ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
ax.set_ylabel('IC (Pearson)')
plt.tight_layout()
plt.show()

aligned['quintile'] = pd.qcut(aligned['signal'], 5, labels=False, duplicates='drop')
qmeans = aligned.groupby('quintile')['fwd_ret'].mean() * 12
fig2, ax2 = plt.subplots(figsize=(8, 4))
qmeans.plot(kind='bar', ax=ax2, color='steelblue',
            title='Signal Quintile — Mean Annualised Forward Return')
ax2.set_xlabel('Quintile (1=low, 5=high)')
ax2.set_ylabel('Annualised Return')
plt.tight_layout()
plt.show()

## 7. Export


In [ ]:
signal.to_frame('signal').to_parquet(
    BASE_PATH + 'data/processed/signals.parquet'
)


## Limitations

- Rolling z-score is symmetric -- asymmetric distributions may bias.
- Lag of 1 assumes monthly release cadence.
